In [0]:
# COMMAND ----------
import os
import requests
import uuid as _uuid
from pyspark.sql.functions import current_timestamp, lit
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from datetime import datetime

# 1. Parameters
dbutils.widgets.text("wfp_file_path", "/Volumes/workspace/default/spi_data/samples/wfpvam_foodprices.csv")
wfp_file_path = dbutils.widgets.get("wfp_file_path")
start_time = datetime.now()

# 2. Secure API Fetch
wfp_api_url = "https://data.humdata.org/dataset/999b66ee-5c62-4afc-bfab-d72b22fcebc2/resource/71542fba-3837-4d7a-b9c1-4b13a7c36a32/download/wfpvam_foodprices.csv"
print(f"Fetching latest WFP data from HDX API to {wfp_file_path}...")

# Mask the script as a standard web browser to prevent the API from blocking Databricks
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/114.0.0.0 Safari/537.36"
}

try:
    # Stream=True ensures the 200MB file doesn't overload cluster memory
    response = requests.get(wfp_api_url, headers=headers, stream=True, timeout=120)
    response.raise_for_status() 

    # Write in 8KB chunks directly to the Unity Catalog Volume
    with open(wfp_file_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)
    print("Successfully downloaded and saved WFP dataset.")
except Exception as e:
    print(f"API Download Failed: {e}. The pipeline will attempt to proceed using the existing cached file.")

# 3. Redefined locally for session isolation
wfp_schema = StructType([
    StructField("adm0_id", IntegerType(), True), StructField("adm0_name", StringType(), True),
    StructField("adm1_id", IntegerType(), True), StructField("adm1_name", StringType(), True),
    StructField("mkt_id", IntegerType(), True), StructField("mkt_name", StringType(), True),
    StructField("cm_id", IntegerType(), True), StructField("cm_name", StringType(), True),
    StructField("cur_id", IntegerType(), True), StructField("cur_name", StringType(), True),
    StructField("pt_id", IntegerType(), True), StructField("pt_name", StringType(), True),
    StructField("um_id", IntegerType(), True), StructField("um_name", StringType(), True),
    StructField("mp_month", IntegerType(), True), StructField("mp_year", IntegerType(), True),
    StructField("mp_price", DoubleType(), True), StructField("mp_commoditysource", StringType(), True)
])

Fetching latest WFP data from HDX API to /Volumes/workspace/default/spi_data/samples/wfpvam_foodprices.csv...
API Download Failed: HTTPSConnectionPool(host='data.humdata.org', port=443): Max retries exceeded with url: /dataset/999b66ee-5c62-4afc-bfab-d72b22fcebc2/resource/71542fba-3837-4d7a-b9c1-4b13a7c36a32/download/wfpvam_foodprices.csv (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0xff9d0c442b40>: Failed to resolve 'data.humdata.org' ([Errno -3] Temporary failure in name resolution)")). The pipeline will attempt to proceed using the existing cached file.


In [0]:
# COMMAND ----------
# CELL 2: Ingest and Write
# mergeSchema handles upstream source changes dynamically without failing the pipeline.
file_name = os.path.basename(wfp_file_path)

try:
    df_wfp = spark.read.format("csv").option("header", "true").schema(wfp_schema).load(wfp_file_path) \
        .withColumn("load_timestamp", current_timestamp())

    # Verification
    print("--- Raw WFP Preview ---")
    display(df_wfp.limit(3))

    df_wfp.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("default.bronze_wfp_raw")

    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-WFP', '{file_name}', '{start_time}', current_timestamp(), 'Success', {df_wfp.count()}, '')")
    print(f"Successfully processed {file_name} into Bronze layer.")
    
except Exception as e:
    safe_error = str(e)[0:200].replace("'", "")
    spark.sql(f"INSERT INTO default.pipeline_execution_logs VALUES ('{_uuid.uuid4()}', 'Raw-to-Bronze-WFP', '{file_name}', '{start_time}', current_timestamp(), 'Failure', 0, '{safe_error}')")
    raise e

--- Raw WFP Preview ---


adm0_id,adm0_name,adm1_id,adm1_name,mkt_id,mkt_name,cm_id,cm_name,cur_id,cur_name,pt_id,pt_name,um_id,um_name,mp_month,mp_year,mp_price,mp_commoditysource,load_timestamp
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,1,2014,50.0,null,2026-10-08T05:39:49.527Z
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,2,2014,50.0,null,2026-10-08T05:39:49.527Z
null,Afghanistan,272,Badakhshan,266,Fayzabad,55,Bread - Retail,null,AFN,15,Retail,5,KG,3,2014,50.0,null,2026-10-08T05:39:49.527Z
